# MIMIC-IV & MIMIC-CXR Exploratory Analysis

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
sys.path.append(str(PROJECT_ROOT))

import config
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 1000)

Matplotlib is building the font cache; this may take a moment.


## Raw data files

In [2]:
def check_directory(path):
    if not path.exists():
        print(f"{path.name}: Directory not found")
        return
    files = sorted(path.glob("*.csv*"))
    if not files:
        print(f"{path.name}: No CSV files found")
        return
    print(f"Files in {path.name}/ ({len(files)} files):")
    for f in files:
        size_mb = f.stat().st_size / (1024 * 1024)
        print(f"  {f.name:<30} {size_mb:8.2f} MB")

check_directory(config.MIMIC_IV_HOSP_DIR)
print()
check_directory(config.MIMIC_IV_ICU_DIR)
print()
check_directory(config.MIMIC_CXR_DIR)

Files in hosp/ (22 files):
  admissions.csv.gz                 19.00 MB
  d_hcpcs.csv.gz                     0.41 MB
  d_icd_diagnoses.csv.gz             0.84 MB
  d_icd_procedures.csv.gz            0.56 MB
  d_labitems.csv.gz                  0.01 MB
  diagnoses_icd.csv.gz              32.01 MB
  drgcodes.csv.gz                    9.29 MB
  emar.csv.gz                      773.72 MB
  emar_detail.csv.gz               713.50 MB
  hcpcsevents.csv.gz                 2.06 MB
  labevents.csv.gz                2472.79 MB
  microbiologyevents.csv.gz        112.19 MB
  omr.csv.gz                        42.03 MB
  patients.csv.gz                    2.70 MB
  pharmacy.csv.gz                  501.35 MB
  poe.csv.gz                       635.71 MB
  poe_detail.csv.gz                 52.71 MB
  prescriptions.csv.gz             578.21 MB
  procedures_icd.csv.gz              7.42 MB
  provider.csv.gz                    0.12 MB
  services.csv.gz                    8.17 MB
  transfers.csv.gz          

## ICU admissions

In [3]:
icustays_path = config.MIMIC_IV_ICU_DIR / "icustays.csv.gz"
if not icustays_path.exists():
    icustays_path = config.MIMIC_IV_DIR / "icustays.csv.gz"

if icustays_path.exists():
    df_icu = pd.read_csv(icustays_path)
    print(f"Total ICU stays: {len(df_icu):,}")
    print(f"Unique patients: {df_icu['subject_id'].nunique():,}")
    display(df_icu.head())
    
    # Summary of ICU length of stay
    print("\nLength of stay (days):")
    print(df_icu['los'].describe())
else:
    print(f"File not found: {icustays_path}")

Total ICU stays: 94,458
Unique patients: 65,366


,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los
0,10000032,29079034,39553978,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2180-07-23 14:00:00,2180-07-23 23:50:47,0.410266
1,10000690,25860671,37081114,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2150-11-02 19:37:00,2150-11-06 17:03:17,3.893252
2,10000980,26913865,39765666,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2189-06-27 08:42:00,2189-06-27 20:38:27,0.497535
3,10001217,24597018,37067082,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2157-11-20 19:18:02,2157-11-21 22:08:00,1.118032
4,10001217,27703517,34592300,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2157-12-19 15:42:24,2157-12-20 14:27:41,0.948113



Length of stay (days):
count    94444.000000
mean         3.630025
std          5.402474
min          0.001250
25%          1.096212
50%          1.965648
75%          3.862575
max        226.403079
Name: los, dtype: float64


## Item IDs for vitals and labs

In [4]:
# Vitals item IDs from config
for vital, itemids in config.VITAL_ITEM_IDS.items():
    print(f"{vital:<15}: {itemids}")

print()
# Labs item IDs from config
for lab, itemids in config.LAB_ITEM_IDS.items():
    print(f"{lab:<15}: {itemids}")

heart_rate     : [220045]
map            : [220052, 220181, 225312]
spo2           : [220277]
temperature    : [223761, 223762]
resp_rate      : [220210, 224690]

lactate        : [50813, 52442, 53154]
platelets      : [51265]
creatinine     : [50912]
bilirubin      : [50885]
wbc            : [51300, 51301]


## Chartevents schema sample

In [5]:
# Read small sample to verify column names
chart_path = config.MIMIC_IV_ICU_DIR / "chartevents.csv.gz"
if not chart_path.exists():
    chart_path = config.MIMIC_IV_DIR / "chartevents.csv.gz"

if chart_path.exists():
    df_chart = pd.read_csv(chart_path, nrows=100)
    display(df_chart.head())
else:
    print(f"File not found: {chart_path}")

,subject_id,hadm_id,stay_id,caregiver_id,charttime,storetime,itemid,value,valuenum,valueuom,warning
0,10000032,29079034,39553978,18704,2180-07-23 12:36:00,2180-07-23 14:45:00,226512,39.4,39.4,kg,0
1,10000032,29079034,39553978,18704,2180-07-23 12:36:00,2180-07-23 14:45:00,226707,60,60.0,Inch,0
2,10000032,29079034,39553978,18704,2180-07-23 12:36:00,2180-07-23 14:45:00,226730,152,152.0,cm,0
3,10000032,29079034,39553978,18704,2180-07-23 14:00:00,2180-07-23 14:18:00,220048,SR (Sinus Rhythm),NaN,NaN,0
4,10000032,29079034,39553978,18704,2180-07-23 14:00:00,2180-07-23 14:18:00,224642,Oral,NaN,NaN,0


## Chest X-Ray metadata

In [6]:
cxr_meta_path = config.MIMIC_CXR_DIR / "mimic-cxr-2.0.0-metadata.csv.gz"
if not cxr_meta_path.exists():
    cxr_meta_path = config.MIMIC_CXR_DIR / "mimic-cxr-2.0.0-metadata.csv"

if cxr_meta_path.exists():
    df_cxr = pd.read_csv(cxr_meta_path)
    print(f"Total CXR records: {len(df_cxr):,}")
    print(f"Unique patients: {df_cxr['subject_id'].nunique():,}")
    display(df_cxr.head())
    
    # View position distribution
    print("\nView positions:")
    print(df_cxr['ViewPosition'].value_counts(dropna=False))
    
    frontal = df_cxr['ViewPosition'].isin(['PA', 'AP']).sum()
    print(f"\nFrontal scans (PA/AP): {frontal:,} ({frontal / len(df_cxr) * 100:.1f}%)")
else:
    print(f"File not found: {cxr_meta_path}")

Total CXR records: 377,110
Unique patients: 65,379


,dicom_id,subject_id,study_id,PerformedProcedureStepDescription,ViewPosition,Rows,Columns,StudyDate,StudyTime,ProcedureCodeSequence_CodeMeaning,ViewCodeSequence_CodeMeaning,PatientOrientationCodeSequence_CodeMeaning
0,02aa804e-bde0afdd-112c0b34-7bc16630-4e384014,10000032,50414267,CHEST (PA AND LAT),PA,3056,2544,21800506,213014.531,CHEST (PA AND LAT),postero-anterior,Erect
1,174413ec-4ec4c1f7-34ea26b7-c5f994f8-79ef1962,10000032,50414267,CHEST (PA AND LAT),LATERAL,3056,2544,21800506,213014.531,CHEST (PA AND LAT),lateral,Erect
2,2a2277a9-b0ded155-c0de8eb9-c124d10e-82c5caab,10000032,53189527,CHEST (PA AND LAT),PA,3056,2544,21800626,165500.312,CHEST (PA AND LAT),postero-anterior,Erect
3,e084de3b-be89b11e-20fe3f9f-9c8d8dfe-4cfd202c,10000032,53189527,CHEST (PA AND LAT),LATERAL,3056,2544,21800626,165500.312,CHEST (PA AND LAT),lateral,Erect
4,68b5c4b1-227d0485-9cc38c3f-7b84ab51-4b472714,10000032,53911762,CHEST (PORTABLE AP),AP,2705,2539,21800723,80556.875,CHEST (PORTABLE AP),antero-posterior,NaN



View positions:
ViewPosition
AP                147173
PA                 96161
LATERAL            82853
LL                 35133
NaN                15769
PA LLD                 4
LAO                    3
RAO                    3
AP AXIAL               2
AP LLD                 2
XTABLE LATERAL         2
AP RLD                 2
SWIMMERS               1
PA RLD                 1
LPO                    1
Name: count, dtype: int64

Frontal scans (PA/AP): 243,334 (64.5%)
